# Step 3: Feature Engineering and Encoding

In this step, we prepare the data for predictive modeling by:

- Creating new risk-related features
- Encoding categorical variables
- Preparing the final modeling dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("insurance_clean.csv")

df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 1. Create Age Groups

Age is often used in insurance pricing bands rather than as a continuous variable.

In [2]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[18, 30, 40, 50, 60, 70],
    labels=['18-30', '31-40', '41-50', '51-60', '61-70'],
    include_lowest=True
)

df[['age', 'age_group']].head()

,age,age_group
0,19,18-30
1,18,18-30
2,28,18-30
3,33,31-40
4,32,31-40


## 2. Create BMI Risk Categories

BMI categories help identify different health risk levels.

In [3]:
df['bmi_category'] = pd.cut(
    df['bmi'],
    bins=[0, 18.5, 25, 30, 100],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
)

df[['bmi', 'bmi_category']].head()

,bmi,bmi_category
0,27.900,Overweight
1,33.770,Obese
2,33.000,Obese
3,22.705,Normal
4,28.880,Overweight


## 3. Create High-Risk BMI Indicator

Customers with BMI ≥ 30 are considered obese and may represent higher insurance risk.

In [4]:
df['high_bmi'] = np.where(df['bmi'] >= 30, 1, 0)

df[['bmi', 'high_bmi']].head()

,bmi,high_bmi
0,27.900,0
1,33.770,1
2,33.000,1
3,22.705,0
4,28.880,0


## 4. Create Smoker-BMI Interaction Feature

This feature captures the combined effect of smoking and obesity.

In [5]:
df['smoker_binary'] = df['smoker'].map({
    'no': 0,
    'yes': 1
})

df['smoker_bmi_interaction'] = (
    df['smoker_binary'] * df['bmi']
)

df[['smoker', 'bmi', 'smoker_bmi_interaction']].head()

,smoker,bmi,smoker_bmi_interaction
0,yes,27.900,27.9
1,no,33.770,0.0
2,no,33.000,0.0
3,no,22.705,0.0
4,no,28.880,0.0


## 5. Create Family Size Feature

Family size includes the policyholder and all dependents.

In [6]:
df['family_size'] = df['children'] + 1

df[['children', 'family_size']].head()

,children,family_size
0,0,1
1,1,2
2,3,4
3,0,1
4,0,1


## 6. Log Transformation of Charges

Insurance costs are highly skewed. Applying a logarithmic transformation often improves model performance.

In [7]:
df['log_charges'] = np.log(df['charges'])

df[['charges', 'log_charges']].head()

,charges,log_charges
0,16884.92400,9.734176
1,1725.55230,7.453302
2,4449.46200,8.400538
3,21984.47061,9.998092
4,3866.85520,8.260197


## 7. Encode Binary Variables

Convert categorical variables into numerical format.

In [8]:
df['sex'] = df['sex'].map({
    'female': 0,
    'male': 1
})

df['smoker'] = df['smoker'].map({
    'no': 0,
    'yes': 1
})

df[['sex', 'smoker']].head()

,sex,smoker
0,0,1
1,1,0
2,1,0
3,1,0
4,1,0


## 8. One-Hot Encode Multi-Class Variables

Convert region, age group, and BMI category into dummy variables.

In [9]:
df_model = pd.get_dummies(
    df,
    columns=[
        'region',
        'age_group',
        'bmi_category'
    ],
    drop_first=True
)

## 9. Review Final Dataset

In [10]:
print("Final Dataset Shape:")
print(df_model.shape)

df_model.head()

Final Dataset Shape:
(1337, 21)


,age,sex,bmi,children,smoker,charges,high_bmi,smoker_binary,smoker_bmi_interaction,family_size,...,region_northwest,region_southeast,region_southwest,age_group_31-40,age_group_41-50,age_group_51-60,age_group_61-70,bmi_category_Normal,bmi_category_Overweight,bmi_category_Obese
0,19,0,27.900,0,1,16884.92400,0,1,27.9,1,...,0,0,1,0,0,0,0,0,1,0
1,18,1,33.770,1,0,1725.55230,1,0,0.0,2,...,0,1,0,0,0,0,0,0,0,1
2,28,1,33.000,3,0,4449.46200,1,0,0.0,4,...,0,1,0,0,0,0,0,0,0,1
3,33,1,22.705,0,0,21984.47061,0,0,0.0,1,...,1,0,0,1,0,0,0,1,0,0
4,32,1,28.880,0,0,3866.85520,0,0,0.0,1,...,1,0,0,1,0,0,0,0,1,0


## 10. Check Data Types

In [11]:
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1337 entries, 0 to 1336
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   age                      1337 non-null   int64  
 1   sex                      1337 non-null   int64  
 2   bmi                      1337 non-null   float64
 3   children                 1337 non-null   int64  
 4   smoker                   1337 non-null   int64  
 5   charges                  1337 non-null   float64
 6   high_bmi                 1337 non-null   int32  
 7   smoker_binary            1337 non-null   int64  
 8   smoker_bmi_interaction   1337 non-null   float64
 9   family_size              1337 non-null   int64  
 10  log_charges              1337 non-null   float64
 11  region_northwest         1337 non-null   uint8  
 12  region_southeast         1337 non-null   uint8  
 13  region_southwest         1337 non-null   uint8  
 14  age_group_31-40         

## Feature Engineering Summary

The following new features were created:

- age_group
- bmi_category
- high_bmi
- smoker_bmi_interaction
- family_size
- log_charges

The following encoding techniques were applied:

- Binary encoding for sex and smoker
- One-hot encoding for region
- One-hot encoding for age_group
- One-hot encoding for bmi_category

The dataset is now ready for predictive modeling.

## 11. Save the Processed Dataset

Save the engineered and encoded dataset for use in machine learning models.

In [12]:
# Save processed dataset

df_model.to_csv("insurance_model_ready.csv", index=False)

print("Dataset saved successfully as 'insurance_model_ready.csv'")
print("Shape:", df_model.shape)

Dataset saved successfully as 'insurance_model_ready.csv'
Shape: (1337, 21)
